# The Ninefold Tide — Mari's Natural Child Voice Alternative (Chatterbox)

Free private Kaggle GPU run. Uses the original **synthetically generated** Mari voice reference from the existing private supporting-cast Kaggle notebook, never a real child's voice. Tries four expressive variations using MIT-licensed Chatterbox. Voice model weights stored in temporary Kaggle scratch, not saved outputs or the user's mobile connection. All results UNAPPROVED.


In [ ]:
import os,sys,json,subprocess,hashlib,time,traceback,zipfile
from pathlib import Path
from datetime import datetime,timezone
os.environ["HF_HOME"]="/kaggle/temp/ninefold_mari_chatterbox_models"
os.environ["TORCH_HOME"]="/kaggle/temp/ninefold_mari_chatterbox_torch"
root=Path("/kaggle/working/ninefold_mari_original_chatterbox")
root.mkdir(exist_ok=True,parents=True)
status={"character":"Mari","age":6,"model":"ResembleAI/Chatterbox original",
        "approval":"UNAPPROVED_ORIGINAL_SYNTHETIC_VOICE_CANDIDATES",
        "state":"STARTED","completed_steps":[],"candidate_count":0,
        "started_utc":datetime.now(timezone.utc).isoformat(),"files":[]}
def save(): (root/"mari_recast_status.json").write_text(json.dumps(status,indent=2)+"\n")
def sha(b): return hashlib.sha256(b).hexdigest()
save()
try:
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("Free Kaggle CUDA GPU required; no CPU fallback")
    free,total=torch.cuda.mem_get_info()
    if free<5*1024**3:
        raise RuntimeError("Not enough free GPU memory")
    status["completed_steps"].append("GPU preflight")
    source_files=list(Path("/kaggle/input").rglob("mari_voice_01_UNAPPROVED.wav"))
    source_zips=list(Path("/kaggle/input").rglob("ninefold_17_supporting_cast_UNAPPROVED.zip"))
    if len(source_files)==1:
        base_audio=source_files[0].read_bytes()
    elif len(source_zips)==1:
        with zipfile.ZipFile(source_zips[0]) as z:
            base_audio=z.read("mari_voice_01_UNAPPROVED.wav")
    else:
        raise RuntimeError("Missing or duplicate trusted Mari original synthetic reference; check Kaggle kernel_sources")
    ref=root/"mari_original_synthetic_reference_UNAPPROVED.wav"
    ref.write_bytes(base_audio)
    import wave
    with wave.open(str(ref),"rb") as wav:
        if wav.getframerate()<22050 or wav.getsampwidth()!=2 or wav.getnframes()==0:
            raise ValueError("Unexpected reference PCM format")
    status["source_ref_sha256"]=sha(base_audio)
    status["completed_steps"].append("Original Mari synthetic reference loaded from private Kaggle notebook source")
    proc=subprocess.run([sys.executable,"-m","pip","install","-q",
                         "--disable-pip-version-check","chatterbox-tts","soundfile"],
                        capture_output=True,text=True,timeout=700)
    (root/"install_status.txt").write_text(
         "returncode="+str(proc.returncode)+"\n"+(proc.stderr or "")[-3000:])
    if proc.returncode:raise RuntimeError("Free Chatterbox library install failed (check install_status)")
    status["completed_steps"].append("Installed MIT Chatterbox locally on Kaggle")
    from chatterbox.tts import ChatterboxTTS
    import soundfile as sf
    model=ChatterboxTTS.from_pretrained(device="cuda")
    status["completed_steps"].append("Loaded Chatterbox original")
    scripts=[
      ("I can see the step. Please, hold my hand.",0.35,0.30),
      ("Wait. Please don't go. I'm scared.",0.50,0.35),
      ("Can you help me? I can't jump.",0.60,0.30),
      ("You promised you'd hold my hand.",0.42,0.45)
    ]
    for i,(text,exaggeration,cfg) in enumerate(scripts,1):
        seed=20261031+211*i
        torch.manual_seed(seed)
        wav=model.generate(text,audio_prompt_path=str(ref),
                  exaggeration=exaggeration,cfg_weight=cfg)
        signal=wav.squeeze().detach().float().cpu().numpy()
        dest=root/f"mari_chatterbox_{i:02d}_UNAPPROVED.wav"
        sf.write(str(dest),signal,model.sr,subtype="PCM_16")
        with wave.open(str(dest),"rb") as f:
            dur=f.getnframes()/f.getframerate()
        if dur<0.5 or dur>18:raise RuntimeError("Invalid generation duration")
        status["files"].append({"file":dest.name,"text":text,
            "duration_seconds":round(dur,3),"seed":seed,"exaggeration":exaggeration,
            "cfg_weight":cfg,"sha256":sha(dest.read_bytes()),
            "human_reviewed":False,"age_authenticity_approved":False})
        status["candidate_count"]=len(status["files"])
        save()
        print("MARI_CHATTERBOX_CANDIDATE",i,round(dur,2),flush=True)
    status["state"]="ALL_FOUR_ORIGINAL_RETAKES_GENERATED"
except Exception as exc:
    status["state"]="FAILED_OR_PARTIAL"
    status["error"]=type(exc).__name__+": "+str(exc)[:300]
    print("MARI_CHATTERBOX_ERROR",status["error"],flush=True)
finally:
    status["ended_utc"]=datetime.now(timezone.utc).isoformat()
    save()
    archive=Path("/kaggle/working/ninefold_mari_chatterbox_UNAPPROVED.zip")
    with zipfile.ZipFile(archive,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=5) as z:
        for file in root.iterdir():
            if file.suffix.lower() in (".wav",".json",".txt"):
                z.write(file,arcname=file.name)
    print("MARI_CHATTERBOX_FINAL",status["state"],status["candidate_count"],
          archive.stat().st_size,flush=True)
